In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path(r"E:\Projet_Alzheimer")
AUDIT = (
    ROOT / "data" / "05_Manifestes"
    / "audit_20261023_133400_145613"
)

fichiers_requis = [
    AUDIT / "candidats_provisoires.csv",
    AUDIT / "rapprochements_irm_pet_provisoires.csv",
]

print("Python utilisé :", sys.executable)

for fichier in fichiers_requis:
    print(
        "OK" if fichier.is_file() else "INTROUVABLE",
        "—",
        fichier.name,
    )

In [ ]:
from IPython.display import display
import json

# 1. Recharger les candidats et les rapprochements IRM–PET.
candidats = pd.read_csv(
    AUDIT / "candidats_provisoires.csv",
    dtype={"PTID": "string"},
)

paires = pd.read_csv(
    AUDIT / "rapprochements_irm_pet_provisoires.csv",
    dtype={"PTID": "string"},
    parse_dates=["date_irm", "date_pet"],
)

for table in [candidats, paires]:
    table["PTID"] = table["PTID"].str.strip().str.upper()

# Vérifier explicitement le critère IRM–PET.
paires = paires.loc[
    paires["PTID"].isin(candidats["PTID"])
    & (
        (paires["date_irm"] - paires["date_pet"])
        .dt.days.abs() <= 90
    )
].copy()

# 2. Recharger les diagnostics.
fichiers_dx = list(
    (ROOT / "data" / "02_Tables_originales")
    .rglob("All_Subjects_DXSUM_*.csv")
)

if len(fichiers_dx) != 1:
    raise ValueError("Un seul fichier DXSUM est attendu.")

dx = pd.read_csv(
    fichiers_dx[0],
    dtype="string",
    encoding="utf-8-sig",
)

dx["PTID"] = dx["PTID"].str.strip().str.upper()
dx["DIAGNOSIS"] = dx["DIAGNOSIS"].str.strip()
dx["date_dx"] = pd.to_datetime(
    dx["EXAMDATE"].str.strip(),
    format="%Y-%m-%d",
    errors="coerce",
)

dx = dx.loc[
    dx["PTID"].isin(candidats["PTID"])
    & dx["DIAGNOSIS"].isin(["1", "2", "3"])
    & dx["date_dx"].notna()
].copy()

# Écarter les visites présentant des codes contradictoires.
nombre_codes = dx.groupby(
    ["PTID", "date_dx"]
)["DIAGNOSIS"].nunique()

conflits = nombre_codes[nombre_codes > 1].index
index_dx = pd.MultiIndex.from_frame(dx[["PTID", "date_dx"]])

dx = (
    dx.loc[
        ~index_dx.isin(conflits),
        ["PTID", "date_dx", "DIAGNOSIS"],
    ]
    .drop_duplicates()
)

# 3. Diagnostic le plus proche de chaque date IRM, à ±30 jours.
visites_irm = paires[["PTID", "date_irm"]].drop_duplicates()

diagnostics_proches = pd.merge_asof(
    visites_irm.sort_values("date_irm"),
    dx.sort_values("date_dx"),
    by="PTID",
    left_on="date_irm",
    right_on="date_dx",
    direction="nearest",
    tolerance=pd.Timedelta(days=30),
).dropna(subset=["date_dx"])

visites = paires.merge(
    diagnostics_proches,
    on=["PTID", "date_irm"],
    how="inner",
    validate="many_to_one",
)

# 4. Appliquer tous les critères aux mêmes visites.
bilan = visites.groupby("PTID").agg(
    nombre_dates_irm=("date_irm", "nunique"),
    nombre_dates_pet=("date_pet", "nunique"),
    nombre_dates_dx=("date_dx", "nunique"),
    premiere_irm=("date_irm", "min"),
    derniere_irm=("date_irm", "max"),
)

bilan["suivi_jours"] = (
    bilan["derniere_irm"] - bilan["premiere_irm"]
).dt.days

cohorte = bilan.loc[
    (bilan["nombre_dates_irm"] >= 2)
    & (bilan["nombre_dates_pet"] >= 2)
    & (bilan["nombre_dates_dx"] >= 2)
    & (bilan["suivi_jours"] >= 365)
].reset_index()

visites_cohorte = visites.loc[
    visites["PTID"].isin(cohorte["PTID"])
].copy()

print("Participants retrouvés :", len(cohorte))

if len(cohorte) != 359:
    raise ValueError(
        "Le résultat diffère des 359 attendus. "
        "Vérifier les sources avant de poursuivre."
    )

# 5. Enregistrer dans un nouveau dossier, sans écraser l'ancien audit.
execution = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S_%f")
DOSSIER_COHORTE = (
    ROOT / "data" / "05_Manifestes"
    / f"cohorte_359_{execution}"
)
DOSSIER_COHORTE.mkdir(parents=True, exist_ok=False)

cohorte.to_csv(
    DOSSIER_COHORTE / "participants_candidats.csv",
    index=False,
)

visites_cohorte.to_csv(
    DOSSIER_COHORTE / "visites_candidates.csv",
    index=False,
)

(DOSSIER_COHORTE / "identifiants_pour_IDA.txt").write_text(
    ",".join(sorted(cohorte["PTID"])),
    encoding="utf-8",
)

parametres = {
    "statut": "cohorte_candidate_non_validee",
    "traceur": "18F-FDG",
    "ecart_irm_pet_max_jours": 90,
    "ecart_irm_diagnostic_max_jours": 30,
    "minimum_dates_par_source": 2,
    "suivi_irm_minimum_jours": 365,
    "appariement": "plus_proche_avant_ou_apres_non_strictement_un_a_un",
    "audit_source": str(AUDIT),
    "diagnostics_source": fichiers_dx[0].name,
    "controle_qualite_images_effectue": False,
}

(DOSSIER_COHORTE / "criteres_selection.json").write_text(
    json.dumps(parametres, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Dossier enregistré :", DOSSIER_COHORTE)
print(
    "Dates IRM candidates :",
    len(visites_cohorte[["PTID", "date_irm"]].drop_duplicates()),
)
print(
    "Dates PET candidates :",
    len(visites_cohorte[["PTID", "date_pet"]].drop_duplicates()),
)

In [ ]:
from pathlib import Path
import pandas as pd

DOSSIER_COHORTE = Path(
    r"E:\Projet_Alzheimer\data\05_Manifestes"
    r"\cohorte_359_20261024_184212_495491"
)

export = pd.read_csv(
    DOSSIER_COHORTE / "recherche_IRM_359.csv",
    dtype="string",
    encoding="utf-8-sig",
)

requis = {"Subject ID", "Study Date", "Image ID"}
manquants = requis - set(export.columns)

if manquants:
    raise ValueError(
        f"Colonnes manquantes dans l'export : {sorted(manquants)}"
    )

export["PTID"] = export["Subject ID"].str.strip().str.upper()
export["date_irm"] = pd.to_datetime(
    export["Study Date"].str.strip(),
    format="%m/%d/%Y",
    errors="raise",
)

if export[["PTID", "date_irm", "Image ID"]].isna().any().any():
    raise ValueError("Un identifiant ou une date manque dans l'export.")

visites = pd.read_csv(
    DOSSIER_COHORTE / "visites_candidates.csv",
    dtype={"PTID": "string"},
    parse_dates=["date_irm"],
)

dates_cibles = visites[["PTID", "date_irm"]].drop_duplicates()

# Conserver toutes les séries des dates retenues.
series_candidates = export.merge(
    dates_cibles,
    on=["PTID", "date_irm"],
    how="inner",
    validate="many_to_one",
)

dates_trouvees = (
    series_candidates[["PTID", "date_irm"]].drop_duplicates()
)

print("Séries dans l'export :", len(export))
print("Participants dans l'export :", export["PTID"].nunique())
print("Dates cibles retrouvées :", len(dates_trouvees), "/", len(dates_cibles))
print("Séries aux dates cibles :", len(series_candidates))

series_candidates.to_csv(
    DOSSIER_COHORTE / "series_IRM_candidates.csv",
    index=False,
)

In [ ]:
# Vérifier les identifiants avant de compter les séries.
series = series_candidates.copy()
series["Image ID"] = series["Image ID"].str.strip()

print(
    "Identifiants d'image répétés dans l'export filtré :",
    int(series["Image ID"].duplicated().sum()),
)

# Distribution du nombre de séries par participant et date.
nombre_series = (
    series.groupby(["PTID", "date_irm"])["Image ID"].nunique()
)

distribution = (
    nombre_series.value_counts()
    .sort_index()
    .rename_axis("series_par_date")
    .reset_index(name="nombre_de_dates")
)

print("\nNombre de séries disponibles par date :")
display(distribution)

print("\nDescriptions des séries :")
display(
    series["Description"]
    .fillna("NON_RENSEIGNE")
    .value_counts()
    .rename_axis("description")
    .reset_index(name="nombre_de_series")
)

In [ ]:
# Recharger le manifeste enregistré pour pouvoir reprendre plus tard.
series = pd.read_csv(
    DOSSIER_COHORTE / "series_IRM_candidates.csv",
    dtype="string",
)

ids = (
    series["Image ID"]
    .str.strip()
    .str.replace(r"^[Ii]", "", regex=True)
)

if ids.isna().any() or not ids.str.fullmatch(r"\d+").all():
    raise ValueError("Certains identifiants d'image sont absents ou invalides.")

# Un identifiant ne doit apparaître qu'une fois dans les téléchargements.
ids = sorted(set(ids), key=int)

execution = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S_%f")
DOSSIER_LOTS = DOSSIER_COHORTE / f"lots_IRM_{execution}"
DOSSIER_LOTS.mkdir(parents=True, exist_ok=False)

suivi_lots = []

for debut in range(0, len(ids), 100):
    numero = debut // 100 + 1
    lot = ids[debut:debut + 100]
    nom = f"IRM_lot_{numero:03d}"

    (DOSSIER_LOTS / f"{nom}.txt").write_text(
        ",".join(lot),
        encoding="utf-8",
    )

    suivi_lots.append({
        "lot": nom,
        "series_attendues": len(lot),
        "statut": "a_telecharger",
    })

pd.DataFrame(suivi_lots).to_csv(
    DOSSIER_LOTS / "suivi_telechargements.csv",
    index=False,
)

print("Séries uniques :", len(ids))
print("Nombre de lots :", len(suivi_lots))
print("Dossier :", DOSSIER_LOTS)